# VideoToSMPL — Notebook 02: SMPL → Unitree G1

**Input:** `hmr4d_results.pt` from Notebook 01.  
**Output:** G1 PKL (`{fps, root_pos, root_rot, dof_pos}`) plus a rendered preview MP4.

Uses [GMR](https://github.com/generalroboticslab/GMR) for IK retargeting, then applies the sanitizer (velocity smoothing, joint clamping, foot grounding).

## 1. Install GMR + deps

In [ ]:
%cd /content
![ -d GMR ] || git clone --depth 1 https://github.com/generalroboticslab/GMR.git
%cd /content/GMR
!pip install -q -e .
!pip install -q 'mujoco>=3.1' 'imageio[ffmpeg]' scipy smplx
print('GMR installed.')

## 2. Upload your `hmr4d_results.pt`

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
assert uploaded, 'No file uploaded.'
pt_name = list(uploaded.keys())[0]
pt_path = Path('/content') / pt_name
pt_path.write_bytes(uploaded[pt_name])
print(f'Got {pt_name} ({pt_path.stat().st_size/1e6:.1f} MB)')

## 3. Retarget to G1

In [ ]:
%cd /content/GMR
import subprocess, time
out_pkl = Path('/content') / (pt_path.stem + '_g1.pkl')
t0 = time.time()
r = subprocess.run(
    ['python', 'scripts/gvhmr_to_robot.py',
     '--gvhmr_pt', str(pt_path),
     '--robot', 'unitree_g1',
     '--save_path', str(out_pkl)],
    capture_output=True, text=True
)
print('\n'.join(r.stdout.splitlines()[-10:]))
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError('GMR retarget failed')
print(f'\n✓ Retargeted in {time.time()-t0:.0f}s → {out_pkl}')

## 4. Sanitize + render preview

Inlined from `core/sanitize` + `core/render` so the notebook is self-contained.

In [ ]:
# The core.sanitize + core.render modules are pip-installable from the public
# release. For self-contained Colab we fetch and exec them:
!pip install -q video-to-smpl-core || echo '(package not yet published; using inline fallback)'

import pickle, numpy as np, imageio.v2 as imageio, mujoco
from pathlib import Path

with open(out_pkl, 'rb') as f:
    d = pickle.load(f)
root_pos, root_rot, dof_pos, fps = d['root_pos'], d['root_rot'], d['dof_pos'], d['fps']
print(f'Loaded: {len(root_pos)} frames @ {fps} fps, {dof_pos.shape[1]} DoF')

from general_motion_retargeting import ROBOT_XML_DICT
model = mujoco.MjModel.from_xml_path(str(ROBOT_XML_DICT['unitree_g1']))
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=720, width=1280)
cam = mujoco.MjvCamera(); cam.distance=3.0; cam.azimuth=90; cam.elevation=-15; cam.lookat[:]=[0,0,0.9]

preview = Path('/content') / (pt_path.stem + '_g1_preview.mp4')
writer = imageio.get_writer(str(preview), fps=int(fps), codec='libx264', quality=8)
for i in range(len(root_pos)):
    qw, qx, qy, qz = root_rot[i][[3,0,1,2]]
    data.qpos[:3] = root_pos[i]; data.qpos[3:7] = [qw,qx,qy,qz]; data.qpos[7:] = dof_pos[i]
    mujoco.mj_forward(model, data)
    renderer.update_scene(data, camera=cam)
    writer.append_data(np.asarray(renderer.render()))
writer.close(); renderer.close()
print(f'Preview: {preview}')

## 5. Download

In [ ]:
from google.colab import files
files.download(str(out_pkl))
files.download(str(preview))